# 🔺 Delta Lake — Optimize (Manual e Auto)

---

## 1. O Problema dos Small Files

Como vimos, cada `INSERT` individual gera um novo arquivo `.parquet`.  
Em pipelines que inserem dados frequentemente em pequenos lotes, isso leva ao **Small Files Problem**:

```
  INSERT VALUES (1, ...)  →  part-00001.snappy.parquet  (poucos bytes)
  INSERT VALUES (2, ...)  →  part-00002.snappy.parquet  (poucos bytes)
  INSERT VALUES (3, ...)  →  part-00003.snappy.parquet  (poucos bytes)
  ...
  INSERT VALUES (200, .) →  part-00200.snappy.parquet  (poucos bytes)
```

### Por que isso é um problema?

O Spark é otimizado para ler **poucos arquivos grandes** em paralelo, não muitos arquivos pequenos.  
Cada arquivo exige uma tarefa separada no executor — o overhead de inicialização de tarefas domina o tempo total.

```
  200 arquivos de 1 KB:
  ├── 200 tasks no Spark
  ├── 200 aberturas de arquivo no storage
  ├── 200 leituras de metadados do _delta_log
  └── scan time: ~5 segundos

  1 arquivo de 200 KB:
  ├── 1 task no Spark
  ├── 1 abertura de arquivo no storage
  ├── 1 leitura de metadados do _delta_log
  └── scan time: ~70 ms
```

### A solução: `OPTIMIZE`

O comando `OPTIMIZE` compacta os pequenos arquivos em arquivos maiores (bin-packing),  
reduzindo drasticamente o número de arquivos e melhorando a performance de leitura.

```
  Antes do OPTIMIZE:   200 arquivos × 1 KB  =  200 KB total
  Após o OPTIMIZE:       1 arquivo × 200 KB  =  200 KB total

  numberFilesRemoved: 200
  numberFilesAdded:   1
```

> ⚠️ Os arquivos removidos pelo `OPTIMIZE` não são deletados imediatamente do disco —  
> eles ficam como órfãos até que o `VACUUM` seja executado, mantendo o Time Travel disponível.

---

## 2. Otimização Manual

---

### 2.1 Setup — desativando otimizações automáticas

Desativamos as otimizações automáticas para reproduzir o Small Files Problem em sua forma pura.

In [ ]:
%%sql
SET spark.databricks.delta.properties.defaults.autoOptimize.optimizeWrite = false;
SET spark.databricks.delta.properties.defaults.autoOptimize.autoCompact = false;

---
### 2.2 Tabela com small files

Cada iteração do loop executa um `INSERT` separado — gerando **200 commits e 200 arquivos Parquet**.

In [ ]:
%%sql
DROP TABLE IF EXISTS small_files;
CREATE OR REPLACE TABLE small_files(id INT, data STRING);

In [ ]:
%python
# 200 INSERTs individuais → 200 arquivos Parquet + 200 commits no _delta_log
for x in range(1,201):
	query = f'''
	INSERT INTO small_files VALUES ({x}, '{x*x}')
'''
	spark.sql(query)

In [ ]:
%%sql
-- Confirma os 200 commits no _delta_log
DESC HISTORY small_files;

In [ ]:
%%sql
-- A coluna 'numFiles' mostra 200 arquivos Parquet ativos
DESCRIBE DETAIL small_files;

In [ ]:
%%sql
-- Usamos avg() propositalmente: min() e max() seriam otimizados pelas statistics do _delta_log
-- avg() força o Spark a ler todos os 200 arquivos — expondo o custo real dos small files
-- ⏱ Anote o scan time na Spark UI para comparar depois
SELECT avg(id) FROM small_files;

> **Como verificar na Spark UI:**  
> Jobs → SQL/DataFrame → clique no job do `SELECT avg` → "Scan parquet spark..."  
> Você verá: **number of files read = 200** e scan time ≈ 5 segundos

---

### 2.3 Tabela com big files (referência para comparação)

Inserimos os mesmos 200 registros em um único `INSERT` — gerando **1 commit e 1 arquivo Parquet**.

In [ ]:
%%sql
DROP TABLE IF EXISTS big_files;
CREATE OR REPLACE TABLE big_files(id INT, data STRING);

In [ ]:
%python
# 1 INSERT com 200 VALUES → 1 arquivo Parquet + 1 commit no _delta_log
values = ", ".join([f"({x}, '{x*x}')" for x in range(1,201)])
query = f"INSERT INTO big_files VALUES {values}"
spark.sql(query)

In [ ]:
%%sql
-- numFiles = 1 — todos os 200 registros em um único Parquet
DESCRIBE DETAIL big_files;

In [ ]:
-- Confirma visualmente: apenas 1 Parquet no diretório
%fs
ls dbfs:/user/hive/warehouse/big_files

In [ ]:
%%sql
-- ⏱ Anote o scan time — compare com o SELECT avg da small_files
SELECT avg(id)
FROM big_files;

> **Comparação na Spark UI:**
>
> | Tabela | Arquivos lidos | Scan time |
> |---|---|---|
> | `small_files` | 200 | ≈ 5 s |
> | `big_files` | 1 | ≈ 70 ms |
>
> A diferença de performance é causada exclusivamente pela quantidade de arquivos —  
> o volume de dados é idêntico nas duas tabelas.

---

### 2.4 Otimizando a tabela com small files

In [ ]:
%%sql
-- Histórico antes do OPTIMIZE: 200 operações WRITE
DESC HISTORY small_files;

In [ ]:
%%sql
-- Compacta os 200 arquivos pequenos em 1 arquivo maior
OPTIMIZE small_files;

> **Lendo os metrics do retorno:**
> ```
>   numFilesAdded:   1    ← 1 novo Parquet compactado criado
>   numFilesRemoved: 200  ← 200 Parquets pequenos marcados como 'remove'
>   numOutputBytes:  X    ← tamanho do arquivo resultante
> ```
> Os 200 arquivos removidos ainda existem fisicamente até o próximo `VACUUM`.

In [ ]:
%%sql
-- A operação OPTIMIZE aparece como a versão mais recente no histórico
DESC HISTORY small_files;

In [ ]:
%%sql
-- ⏱ Compare o scan time com o resultado anterior da small_files
-- Agora o Spark lê apenas 1 arquivo — performance equivalente à big_files
SELECT avg(id)
FROM small_files;

> **Resultado esperado na Spark UI:**  
> number of files read = **1** | scan time ≈ **70 ms**  
> A tabela continua com os mesmos 200 registros — apenas a estrutura física mudou.

---

## 3. Auto Optimization

Executar `OPTIMIZE` manualmente funciona, mas exige monitoramento.  
O Delta Lake oferece duas features automáticas que previnem o Small Files Problem antes que ele aconteça:

| Feature | O que faz | Quando age |
|---|---|---|
| **Optimize Write** | Reagrupa os dados antes de gravar, produzindo menos arquivos por write | No momento do write |
| **Auto Compact** | Compacta arquivos existentes após um write, se o número exceder um threshold | Após cada write |

```
  Sem Auto Optimization:
  INSERT 1 → parquet 1
  INSERT 2 → parquet 2
  ...                     → acúmulo de small files
  INSERT 50 → parquet 50  → Auto Compact dispara → 1 arquivo compactado

  Com Optimize Write:
  INSERT 1 → Spark reorganiza internamente → parquet 1 (já otimizado)
```

---

### 3.1 Habilitando o Auto Compact

In [ ]:
%%sql
-- Habilita a compactação automática para novas tabelas na sessão
SET spark.databricks.delta.properties.defaults.autoOptimize.autoCompact.enabled = true;

In [ ]:
%python
# Threshold mínimo de arquivos para o Auto Compact disparar
# Por padrão: 50 arquivos → a cada 50 INSERTs, uma compactação automática ocorre
spark.conf.get("spark.databricks.delta.autoCompact.minNumFiles")

> **O que o threshold significa:**  
> O Auto Compact só dispara quando o número de arquivos novos gerados por uma operação  
> atinge o `minNumFiles`. Com o padrão de 50, espera-se compactações a cada ≈ 50 commits.

### 3.2 Demonstrando o Auto Compact

In [ ]:
%%sql
DROP TABLE IF EXISTS small_files_auto;
CREATE OR REPLACE TABLE small_files_auto(id INT, data STRING);

In [ ]:
%python
# Mesmo padrão de 200 INSERTs individuais
# Desta vez, com Auto Compact ativo
for x in range(1,201):
	query = f'''
	INSERT INTO small_files_auto VALUES ({x}, '{x*x}')
'''
	spark.sql(query)

In [ ]:
%%sql
-- Observe que a cada ~50 commits há uma operação de OPTIMIZE intercalada
-- O Delta Lake compactou automaticamente sem nenhuma intervenção manual
DESC HISTORY small_files_auto;

> **Lendo o histórico:**  
> Entre os 200 commits de `WRITE`, você verá entradas de `OPTIMIZE` inseridas automaticamente a cada ≈ 50 operações.  
> Ao final, a tabela terá muito menos arquivos do que a `small_files` sem auto compact.

---

## 4. Resumo — Optimize Manual vs Auto

```
┌──────────────────────────────────────────────────────────────────┐
│                  CICLO DO SMALL FILES PROBLEM                    │
│                                                                  │
│  Muitos INSERTs pequenos                                         │
│        │                                                         │
│        ▼                                                         │
│  N arquivos Parquet pequenos                                     │
│        │                                                         │
│        ▼                                                         │
│  N tasks no Spark por query  →  alta latência de leitura         │
│        │                                                         │
│   ┌────┴────────────────────────┐                                │
│   │                             │                                │
│   ▼                             ▼                                │
│  OPTIMIZE manual          Auto Compact                           │
│  (sob demanda)            (threshold: 50 arquivos)               │
│   │                             │                                │
│   └────────────┬────────────────┘                                │
│                ▼                                                  │
│   N arquivos → 1 arquivo (bin-packing)                           │
│   numFilesRemoved: N | numFilesAdded: 1                          │
│   Arquivos antigos: órfãos até VACUUM                            │
└──────────────────────────────────────────────────────────────────┘
```

### Quando usar cada abordagem

| Abordagem | Quando usar |
|---|---|
| **OPTIMIZE manual** | Tabelas existentes já degradadas; pipelines batch com janela de manutenção definida |
| **Auto Compact** | Pipelines streaming ou micro-batch com inserts contínuos; quando não há controle sobre a frequência de writes |
| **Optimize Write** | Quando os writes já chegam em lotes grandes mas ainda geram múltiplos arquivos por partição |

### Comandos de referência

| Objetivo | Comando |
|---|---|
| Otimizar tabela manualmente | `OPTIMIZE table_name` |
| Ver resultado da otimização | Coluna `metrics` no retorno do `OPTIMIZE` |
| Habilitar Auto Compact globalmente | `SET spark.databricks.delta.properties.defaults.autoOptimize.autoCompact.enabled = true` |
| Verificar threshold do Auto Compact | `spark.conf.get("spark.databricks.delta.autoCompact.minNumFiles")` |
| Desabilitar Optimize Write (demo) | `SET spark.databricks.delta.properties.defaults.autoOptimize.optimizeWrite = false` |
| Ver número de arquivos da tabela | `DESCRIBE DETAIL table_name` → coluna `numFiles` |